# Example Analysis Notebooks

This directory contains example notebooks that pull data from `tlssec` database in order to perform statistical analysis on them and let user explore the resulting tables and graphs interactively!

## Basics

Most notebooks start the same way --- importing `tlssec` module and other necessary data analysis and visualization libraries:

In [ ]:
from sqlmodel import select
import pandas as pd

from tlssec.database.database import Database
import tlssec.core.model as model

Then you configure the database object.
The default option should always work if you are running this from the provided `docker-compose.yaml`.

In [ ]:
db = Database()

### List some recent scan results.

In [ ]:
with db.session:
    df = pd.read_sql(
        select(model.ScanTable)
            .order_by(model.ScanTable.start_time)
            .limit(400),
        db.session.connection(),
    )
# TODO: anonymize before recording example result in notebook
df

### Group by security grade

In [ ]:
def get_grade_from_flat_list_testssl_result(result: list[dict]):
    return [
        item['finding']
        for item in result\
        if item['id'] == 'overall_grade'
    ]

In [ ]:
df['grades'] = df.result.apply(get_grade_from_flat_list_testssl_result)

In [ ]:
grade_counts = df.grades.explode().value_counts(dropna = False).sort_index()
grade_counts

In [ ]:
grade_counts.plot.pie(title = 'percentage of endpoints getting each TLS security grade', autopct='%1.1f%%')

#### What each grade means

| Grade | Meaning |
| --- | --- |
| **A+** to **F** | **Ranks** security posture from best to worst |
| **M** | Domain name **Mismatch** in server certificate. |
| **T** | **Trust issue** --- can not verify server certificate. Are we even scanning the correct server? |
| **NaN** | There are **no grade given** for particular scan. Happens, for example, if the scan couldn't be completed due to network error. |


### Drill down into entries with grade F

In [ ]:
# TODO